# Phase 1: Linear Regression From Scratch with NumPy

In this notebook, we build the mathematical foundations of **Linear Regression** from first principles using pure NumPy:
1. **Feature Matrix ($X$) & Target Vector ($y$)**
2. **The Linear Hypothesis:** $\hat{y} = Xw + b$
3. **Mean Squared Error (MSE) Cost Function:** $J(w, b) = \frac{1}{2m} \sum_{i=1}^m (\hat{y}^{(i)} - y^{(i)})^2$
4. **The Normal Equation (Closed-form Solution):** $w = (X^T X)^{-1} X^T y$
5. **Gradient Descent (Iterative Optimization):** $\frac{\partial J}{\partial w} = \frac{1}{m} X^T (\hat{y} - y)$
6. **Feature Scaling (Z-score Standardization):** Why it prevents gradient explosion/oscillation
7. **Mathematical Equivalence Check:** Proving our scratch implementation yields the exact same weights and predictions as Scikit-Learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import sys
sys.path.append('../src')
from linear_regression_scratch import LinearRegressionScratch

## 1. Generating Controlled Synthetic Real Estate Data
To verify mathematical correctness without being clouded by missing values or noise, we simulate 1,000 houses with known true formula:
$$\text{Price (Lakhs)} = 0.08 \times \text{SqFt} + 12.0 \times \text{BHK} + 15.0 + \epsilon$$

In [ ]:
np.random.seed(42)
m = 1000
sqft = np.random.uniform(500, 3500, (m, 1))
bhk = np.random.randint(1, 6, (m, 1)).astype(float)
noise = np.random.normal(0, 5.0, (m, 1))

price = 0.08 * sqft + 12.0 * bhk + 15.0 + noise
X_raw = np.hstack([sqft, bhk])
y_raw = price

print(f"Feature matrix shape: {X_raw.shape}")
print(f"Target vector shape:  {y_raw.shape}")

## 2. Feature Standardization (Z-score Normalization)
$$z = \frac{x - \mu}{\sigma}$$
Without scaling, SqFt (~2000) dominates BHK (~3), creating an elongated elliptical loss landscape that causes gradient descent to oscillate.

In [ ]:
mu = np.mean(X_raw, axis=0)
sigma = np.std(X_raw, axis=0)
X_scaled = (X_raw - mu) / sigma

print("Mean after scaling:", X_scaled.mean(axis=0).round(4))
print("Std after scaling: ", X_scaled.std(axis=0).round(4))

## 3. Method 1: The Normal Equation

In [ ]:
model_normal = LinearRegressionScratch()
model_normal.fit_normal_equation(X_scaled, y_raw)
y_pred_normal = model_normal.predict(X_scaled)

print("Normal Equation Weights:", model_normal.weights.ravel().round(4))
print(f"Normal Equation Bias:    {model_normal.bias:.4f}")
print(f"Train MSE:               {mean_squared_error(y_raw, y_pred_normal):.4f}")
print(f"Train R2 Score:          {r2_score(y_raw, y_pred_normal):.4f}")

## 4. Method 2: Gradient Descent Optimization

In [ ]:
model_gd = LinearRegressionScratch(learning_rate=0.05, n_iterations=800)
model_gd.fit_gradient_descent(X_scaled, y_raw, verbose=True)
y_pred_gd = model_gd.predict(X_scaled)

print("\nGradient Descent Weights:", model_gd.weights.ravel().round(4))
print(f"Gradient Descent Bias:    {model_gd.bias:.4f}")
print(f"Train MSE:                {mean_squared_error(y_raw, y_pred_gd):.4f}")
print(f"Train R2 Score:           {r2_score(y_raw, y_pred_gd):.4f}")

## 5. Loss Curve Visualization

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(model_gd.cost_history, color='#2563eb', linewidth=2)
plt.title('Cost Function J(w, b) Convergence over Iterations', fontsize=12, fontweight='bold')
plt.xlabel('Iteration')
plt.ylabel('Cost (MSE / 2)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 6. Benchmarking Against Scikit-Learn

In [ ]:
sklearn_reg = SklearnLinearRegression()
sklearn_reg.fit(X_scaled, y_raw)
y_pred_sk = sklearn_reg.predict(X_scaled)

max_w_diff = np.max(np.abs(model_gd.weights.ravel() - sklearn_reg.coef_.ravel()))
b_diff = abs(model_gd.bias - float(np.ravel(sklearn_reg.intercept_)[0]))

print(f"Difference in weights: {max_w_diff:.8f}")
print(f"Difference in bias:    {b_diff:.8f}")
print("Exact Match:", max_w_diff < 1e-4 and b_diff < 1e-4)